In [ ]:
!pip install biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 51.7 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import os
import glob
import subprocess
import multiprocessing
from Bio import SeqIO

# GTDB Database

In [ ]:
cols_to_use = ['accession', 'gtdb_taxonomy', 'checkm_completeness', 'checkm_contamination', 'ncbi_assembly_level']
gtdb_df = pd.read_csv('bac120_metadata.tsv', sep='\t', usecols=cols_to_use)
print(gtdb_df)

Filtering the E.Coli data.

In [ ]:
qc_filter = (gtdb_df['checkm_completeness'] >= 95.0) & \
            (gtdb_df['checkm_contamination'] <= 5.0) & \
            (gtdb_df['ncbi_assembly_level'].isin(['Complete Genome', 'Chromosome']))
filtered_gtdb = gtdb_df[qc_filter].copy()
target_spec = ['s__Escherichia coli']
filtered_gtdb = filtered_gtdb[filtered_gtdb['gtdb_taxonomy'].str.contains('|'.join(target_spec))]
print(filtered_gtdb)

Save accession as text file.

In [ ]:
filtered_gtdb['accession'] = filtered_gtdb['accession'].str.replace(r'^(RS_|GB_)', '', regex=True)
accessions = filtered_gtdb['accession'].tolist()
# Save accessions to a text file for downloading
with open("ecoli_accessions.txt", "w") as f:
    for acc in accessions:
        f.write(f"{acc}\n")

print(f"Found {len(accessions)} E.Coli genomes in GTDB.")

Download the genome files from NCBI.

In [ ]:
#@markdown Download the native Linux x86-64 binary from NCBI
!curl -o datasets https://ftp.ncbi.nlm.nih.gov/pub/datasets/command-line/v2/linux-amd64/datasets

#@markdown Make the file executable
!chmod +x datasets

#@markdown Download the genome sequences based on your accessions file
!./datasets download genome accession --inputfile ecoli_accessions.txt --include genome --filename drive/MyDrive/ncbi_genomes.zip

Unzip the files.

In [ ]:
#@markdown Unzip the genomes
# Create the target directory
!mkdir -p /content/raw_gtdb_genomes
!unzip -q "/content/drive/MyDrive/ncbi_genomes.zip" -d /content/raw_gtdb_genomes

# Run Prodigal on the genomes

Install prodigal.

In [ ]:
# Install prodigal
!apt-get update && apt-get install prodigal -y

## Run on E.Coli genomes

In [ ]:
input_dir = "/content/raw_gtdb_genomes"
output_dir = "/content/drive/MyDrive/extracted_proteins"
os.makedirs(output_dir, exist_ok=True)

# Find all genomic FASTA files recursively (.fna or .fasta)
genome_files = glob.glob(os.path.join(input_dir, "**", "*_genomic.fna"), recursive=True)

print(f"Found {len(genome_files)} genome files to process.")

for i, genome_path in enumerate(genome_files):
    # Extract the clean filename/accession (e.g., GCA_017542725.1)
    filename = os.path.basename(genome_path)
    accession = "_".join(filename.split("_")[:2])

    print(f"[{i+1}/{len(genome_files)}] Processing {accession}...")

    # Define outputs for this specific genome
    protein_fasta = os.path.join(output_dir, f"{accession}_proteins.faa")
    gene_coordinates_gff = os.path.join(output_dir, f"{accession}_features.gff")

    # Build the Prodigal command
    command = [
        "prodigal",
        "-i", genome_path,
        "-a", protein_fasta,
        "-o", gene_coordinates_gff,
        "-p", "single"
    ]

    try:
        # Execute Prodigal, capture standard error if it fails
        subprocess.run(command, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE)
    except subprocess.CalledProcessError as e:
        print(f"Error processing {accession}: {e.stderr.decode().strip()}")
        continue

print(f"\nAll genomes successfully processed! Protein files saved to '{output_dir}/'")

## Run Prodigal on viral genomes

In [ ]:
# Inputs
multi_fasta = "/content/sequences.fasta"

split_genome_dir = "/content/phage_genomes/split_fastas"
protein_output_dir = "/content/phage_genomes/proteins"
gff_output_dir = "/content/phage_genomes/gff"

# Make the directories
os.makedirs(split_genome_dir, exist_ok=True)
os.makedirs(protein_output_dir, exist_ok=True)
os.makedirs(gff_output_dir, exist_ok=True)

# Split the multi-FASTA into individual genome files
print("Splitting multi-FASTA into individual genome files...")

genome_paths_list = []

for record in SeqIO.parse(multi_fasta, "fasta"):

    # Clean accession/name
    accession = record.id.split(".")[0]

    genome_path = os.path.join(
        split_genome_dir,
        f"{accession}.fna"
    )

    SeqIO.write(record, genome_path, "fasta")

    genome_paths_list.append((accession, genome_path))

print(f"Created {len(genome_paths_list)} individual genome FASTA files.")

# Run prodigal with multi-processing
print("\nRunning prodigal on phage genomes using multiprocessing...\n")

def process_phage_genome_file(accession, genome_path, protein_output_dir, gff_output_dir):
    """Worker function to process a single phage genome file with prodigal via subprocess."""
    protein_fasta = os.path.join(
        protein_output_dir,
        f"{accession}_proteins.faa"
    )

    gff_output = os.path.join(
        gff_output_dir,
        f"{accession}.gff"
    )

    try:
        # Construct the prodigal command
        command = [
            "prodigal",
            "-i", genome_path,
            "-a", protein_fasta,
            "-o", gff_output,
            "-p", "meta"
        ]

        # Execute the command
        result = subprocess.run(command, capture_output=True, text=True, check=True)

        return f"Successfully processed {accession}"
    except subprocess.CalledProcessError as e:
        return f"Error processing {accession} with Prodigal (exit code {e.returncode}): {e.stderr}"
    except Exception as e:
        return f"Unexpected error processing {accession}: {e}"

# Prepare arguments for multiprocessing
tasks = [(accession, genome_path, protein_output_dir, gff_output_dir) for accession, genome_path in genome_paths_list]

# Use multiprocessing Pool to parallelize the processing
with multiprocessing.Pool() as pool:
    results = list(pool.starmap(process_phage_genome_file, tasks))

# Print results for each processed genome
for result in results:
    print(result)

print("\nFinished processing all phage genomes.")

# Run ESM-2

In [ ]:
# Install fair-esm and torch
!pip install fair-esm torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.1/93.1 kB 8.0 MB/s eta 0:00:00


Load the pre-trained model.

In [ ]:
import torch
import esm
from Bio import SeqIO

# Load ESM-2 model
model, alphabet = esm.pretrained.esm2_t33_650M_UR50D()

# Move model to GPU if available
if torch.cuda.is_available():
    model = model.cuda()
    print("ESM-2 model moved to GPU.")
else:
    print("No GPU available, ESM-2 model running on CPU.")

model.eval()  # Set the model to evaluation mode
batch_converter = alphabet.get_batch_converter()

print("ESM-2 model loaded successfully.")

Downloading: "https://dl.fbaipublicfiles.com/fair-esm/models/esm2_t33_650M_UR50D.pt" to /root/.cache/torch/hub/checkpoints/esm2_t33_650M_UR50D.pt
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/regression/esm2_t33_650M_UR50D-contact-regression.pt" to /root/.cache/torch/hub/checkpoints/esm2_t33_650M_UR50D-contact-regression.pt
ESM-2 model moved to GPU.
ESM-2 model loaded successfully.


Define the function to feed genomes to esm2.

In [ ]:
import numpy as np
import torch
import os

def generate_esm2_embeddings(protein_fasta_path, output_embedding_dir, sequence_batch_size=1, max_sequence_length=1024):
    """Generates ESM-2 embeddings with aggressive memory management."""
    filename = os.path.basename(protein_fasta_path)
    accession = filename.split('_proteins.faa')[0]
    output_path = os.path.join(output_embedding_dir, f'{accession}_esm2_embeddings.npy')

    if os.path.exists(output_path):
        return

    sequences_to_process = []
    for record in SeqIO.parse(protein_fasta_path, "fasta"):
        cleaned_seq = str(record.seq).replace('*', '')
        if cleaned_seq:
            if len(cleaned_seq) > max_sequence_length:
                cleaned_seq = cleaned_seq[:max_sequence_length]
            sequences_to_process.append((record.id, cleaned_seq))

    if not sequences_to_process:
        return

    all_sequence_representations = []
    device = next(model.parameters()).device

    for i in range(0, len(sequences_to_process), sequence_batch_size):
        batch_sequences = sequences_to_process[i:i + sequence_batch_size]

        # 1. Convert to tokens
        _, _, batch_tokens = batch_converter(batch_sequences)
        batch_tokens = batch_tokens.to(device)

        # 2. Forward pass
        with torch.no_grad():
            results = model(batch_tokens, repr_layers=[33], return_contacts=False)
            token_representations = results["representations"][33]

        # 3. Extract and average
        for j, (_, seq) in enumerate(batch_sequences):
            embedding = token_representations[j, 1 : len(seq) + 1].mean(0)
            all_sequence_representations.append(embedding.cpu().numpy())

        # 4. CRITICAL: Explicitly clear GPU memory
        del batch_tokens
        del results
        del token_representations
        torch.cuda.empty_cache()

    if all_sequence_representations:
        all_embeddings = np.array(all_sequence_representations)
        np.save(output_path, all_embeddings)
        print(f"Generated embeddings for {accession} ({len(all_sequence_representations)} proteins)")

## Run the functions on bacterial data

In [ ]:
import os
import glob
output_protein_dir = "/content/drive/MyDrive/extracted_proteins"
output_embedding_dir = "/content/bacterial_esm2_embeddings"
os.makedirs(output_embedding_dir, exist_ok=True)

protein_fasta_files = glob.glob(os.path.join(output_protein_dir, "*_proteins.faa"))

print(f"Found {len(protein_fasta_files)} bacterial protein FASTA files to process for ESM-2 embeddings.")

# --- MODIFICATION START ---
# Reduce the number of files to process to avoid out-of-memory issues.
# You can adjust this number based on your available RAM.
n_bacterial_files_to_process = 10 # Example: process 500 files
if len(protein_fasta_files) > n_bacterial_files_to_process:
    protein_fasta_files = protein_fasta_files[:n_bacterial_files_to_process]
    print(f"Processing a subset of {len(protein_fasta_files)} bacterial protein FASTA files.")
# --- MODIFICATION END ---

for protein_file in protein_fasta_files:
    generate_esm2_embeddings(protein_file, output_embedding_dir)

print("\nAll bacterial protein files processed for ESM-2 embeddings.")

## Now run the function on viral data

In [ ]:
viral_output_protein_dir = "/content/phage_genomes/phage_genomes/proteins"
viral_output_embedding_dir = "/content/viral_esm2_embeddings"
os.makedirs(viral_output_embedding_dir, exist_ok=True)

viral_protein_fasta_files = glob.glob(os.path.join(viral_output_protein_dir, "*_proteins.faa"))

print(f"Found {len(viral_protein_fasta_files)} viral protein FASTA files to process for ESM-2 embeddings.")

# --- MODIFICATION START ---
# Reduce the number of files to process to avoid out-of-memory issues.
# You can adjust this number based on your available RAM.
n_viral_files_to_process = 100 # Example: process 100 files
if len(viral_protein_fasta_files) > n_viral_files_to_process:
    viral_protein_fasta_files = viral_protein_fasta_files[:n_viral_files_to_process]
    print(f"Processing a subset of {len(viral_protein_fasta_files)} viral protein FASTA files.")
# --- MODIFICATION END ---

for protein_file in viral_protein_fasta_files:
    generate_esm2_embeddings(protein_file, viral_output_embedding_dir)

print("\nAll viral protein files processed for ESM-2 embeddings.")